In [68]:
import math

class Value:
    def __init__(self, data, _children=()):
        self.data = data
        self.grad = 0.0
        self._prev = set(_children)
        self._backward = lambda: None
        
    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,))

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def __neg__(self):         
        return self * -1

    def __sub__(self, other):       
        return self + (-other)

    def __radd__(self, other):      # other + self
        return self + other

    def __rmul__(self, other):      # other * self
        return self * other

    def __rsub__(self, other):      # other - self
        return other + (-self)

    def __pow__(self, other):    
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self,))
    
        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
    
        return out
    def backward(self):
        topo = []

        visited = set()

        def build_topo(v):

            if v not in visited:
                visited.add(v)

            for child in v._prev:
                build_topo(child)

            topo.append(v)

        build_topo(self)

        self.grad = 1.0

        for v in reversed(topo):
            v._backward()

In [69]:
import random

class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]

In [70]:
class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

In [71]:
class MLP:

    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return[p for layer in self.layers for p in layer.parameters()]

In [72]:
# Multi layer perception MLP

In [73]:
n = MLP(3, [4, 4, 1])

xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]

for step in range(50):
    # forward pass
    ypred = [n(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
    
    # zero gradients
    for p in n.parameters():
        p.grad = 0.0
    
    # backward pass
    loss.backward()
    
    # update weights
    for p in n.parameters():
        p.data += -0.05 * p.grad
    
    print(step, loss.data)

0 3.766130054471344
1 3.13155842099593
2 5.776585571058426
3 3.9485312914936648
4 3.726514168660607
5 3.9928550817405712
6 3.9765634049204754
7 3.964385740827021
8 3.9056873724683765
9 3.506665636101614
10 5.338552444868071
11 3.8172862494303414
12 3.567018499613126
13 2.963949086084509
14 2.6513657409667033
15 2.783181480221431
16 2.5992086136492807
17 1.99446478078462
18 4.103479808387475
19 2.7390607727210075
20 2.6820748143332165
21 2.178881136826637
22 1.08266780785132
23 0.45854038336214675
24 0.31486963046951716
25 0.23800675545183597
26 0.1887704293396873
27 0.15663472770493303
28 0.13261005869448558
29 0.11433007709030307
30 0.10108374247886655
31 0.0900756482685394
32 0.08102615994261765
33 0.07325429824654786
34 0.06666410104213082
35 0.06168312750704061
36 0.05675843259326574
37 0.05249930370931651
38 0.04864932763844508
39 0.04516773825493399
40 0.04204815230196422
41 0.038992432486161924
42 0.03649735917417305
43 0.03412430749426602
44 0.032094346051480996
45 0.0302392994